In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

# ==================================================
# Load saved data
# ==================================================


results_dir = Path(".")

bands_data = pd.read_csv(
    results_dir / "seed_bands.csv",
    parse_dates=["date"],
)

bands_data["date"] = pd.to_datetime(
    bands_data["date"]
)

bands_data["lag"] = pd.to_numeric(
    bands_data["lag"]
).astype(int)

underlying_log = (
    pd.read_csv(
        results_dir
        / "cumulative_log_return.csv",
        index_col=0,
        parse_dates=True,
    )["cumulative_log_return"]
    .dropna()
    .sort_index()
)

underlying_log.index = pd.to_datetime(
    underlying_log.index
)

lags = sorted(
    bands_data["lag"].unique()
)

# ==================================================
# Interactive plot
# ==================================================

def plot_selected_lag(lag):
    lag = int(lag)

    mlp_bands = (
        bands_data[
            (bands_data["model"] == "MLP")
            & (bands_data["lag"] == lag)
        ]
        .set_index("date")
        .sort_index()
    )

    lin_bands = (
        bands_data[
            (bands_data["model"] == "Linear")
            & (bands_data["lag"] == lag)
        ]
        .set_index("date")
        .sort_index()
    )

    if mlp_bands.empty or lin_bands.empty:
        print(
            f"No saved results for lag {lag}"
        )
        return

    # Rebase the underlying return to the
    # beginning of this test period.
    previous_values = underlying_log[
        underlying_log.index
        < mlp_bands.index[0]
    ]

    baseline = (
        previous_values.iloc[-1]
        if len(previous_values) > 0
        else 0.0
    )

    underlying = (
        underlying_log
        .reindex(mlp_bands.index)
        .dropna()
    )

    underlying_cumulative = (
        np.expm1(
            underlying - baseline
        )
        * 100
    )

    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(11, 9),
        sharex=True,
        sharey=True,
    )
    fig.suptitle('BTH',size=24)

    plot_data = [
        (
            ax1,
            mlp_bands,
            "red",
            "MLP",
        ),
        (
            ax2,
            lin_bands,
            "blue",
            "Linear model",
        ),
    ]

    for ax, bands, color, title in plot_data:
        dates = (
            bands.index.to_pydatetime()
        )

        ax.plot(
            dates,
            bands["mean"].to_numpy(),
            color=color,
            linewidth=1.5,
            label="Seed mean",
        )

        ax.fill_between(
            dates,
            bands[
                "lower_2sigma"
            ].to_numpy(),
            bands[
                "upper_2sigma"
            ].to_numpy(),
            color=color,
            alpha=0.10,
            label=r"$\pm2\sigma$",
        )

        ax.fill_between(
            dates,
            bands[
                "lower_1sigma"
            ].to_numpy(),
            bands[
                "upper_1sigma"
            ].to_numpy(),
            color=color,
            alpha=0.25,
            label=r"$\pm1\sigma$",
        )

        ax.plot(
            underlying_cumulative
            .index
            .to_pydatetime(),
            underlying_cumulative
            .to_numpy(),
            color="black",
            linewidth=1.5,
            label="Underlying",
        )

        ax.axhline(
            0,
            color="grey",
            linewidth=1,
        )

        ax.set_title(
            f"{title} — lookback {lag}"
        )

        ax.set_ylabel(
            "Cumulative return (%)"
        )

        ax.grid(True)
        ax.legend()

    ax2.set_xlabel("Date")

    fig.tight_layout()
    plt.show()

widgets.interact(
    plot_selected_lag,
    lag=widgets.SelectionSlider(
        options=lags,
        value=lags[0],
        description="Lookback:",
        continuous_update=False,
    ),
)

interactive(children=(SelectionSlider(continuous_update=False, description='Lookback:', options=(np.int64(1), …

<function __main__.plot_selected_lag(lag)>

In [2]:


bands_data["date"] = pd.to_datetime(
    bands_data["date"]
)

underlying_log.index = pd.to_datetime(
    underlying_log.index
)

def plot_selected_lag(lag):
    mlp_bands = (
        bands_data[
            (bands_data["model"] == "MLP")
            & (bands_data["lag"] == lag)
        ]
        .set_index("date")
        .sort_index()
    )

    lin_bands = (
        bands_data[
            (bands_data["model"] == "Linear")
            & (bands_data["lag"] == lag)
        ]
        .set_index("date")
        .sort_index()
    )

    previous = underlying_log[
        underlying_log.index
        < mlp_bands.index[0]
    ]

    baseline = (
        previous.iloc[-1]
        if len(previous)
        else 0.0
    )

    underlying = (
        underlying_log
        .reindex(mlp_bands.index)
        .dropna()
    )

    underlying = (
        np.expm1(underlying - baseline)
        * 100
    )

    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(11, 9),
        sharex=True,
        sharey=True,
    )
    

    for ax, bands, color, title in [
        (ax1, mlp_bands, "red", "MLP"),
        (ax2, lin_bands, "blue", "Linear"),
    ]:
        dates = bands.index.to_pydatetime()

        ax.plot(
            dates,
            bands["mean"].to_numpy(),
            color=color,
            label="Seed mean",
        )

        ax.fill_between(
            dates,
            bands["lower_2sigma"].to_numpy(),
            bands["upper_2sigma"].to_numpy(),
            color=color,
            alpha=0.10,
            label=r"$\pm2\sigma$",
        )

        ax.fill_between(
            dates,
            bands["lower_1sigma"].to_numpy(),
            bands["upper_1sigma"].to_numpy(),
            color=color,
            alpha=0.25,
            label=r"$\pm1\sigma$",
        )

        ax.plot(
            underlying.index.to_pydatetime(),
            underlying.to_numpy(),
            color="black",
            label="Underlying",
        )

        ax.set_title(f"{title} — lookback {lag}")
        ax.set_ylabel("Cumulative return (%)")
        ax.grid(True)
        ax.legend()

    fig.tight_layout()
    plt.show()